# Advancement in the Transformer Architecture

### KV Caching

Recall what when we want to predict/generate the next token/word, each heads in the decoding part does the following:

$$Attention(Q,K,V) = \text{Softmax} \left(\frac{Q K^T}{\sqrt{d_k}} + M \right)V$$

This gives us an attention output:

$$O = \begin{bmatrix} o_1 \\ o_1 \\ o_3 \\ \vdots \\ o_n \end{bmatrix}$$

Where each $o_i \in \mathbb{R}^{1 \times d_k}$. Each time we wants to predict the next token, we add another attention output vector $o_{n+1}$ while the rest do not change. As such, recomputing these past attention vectors everytime we want to predict the next token is wasteful. This is where KV-Caching comes in, where we store the Keys and Value vectors from previous tokens in memory. We do not need to cache any Query vectors since all previous Queries pair result in 0 through the masking process. Though effective, its also incredibly memory intensive.

At training time, the Total Memory Access and Total Arithmetic operations for a standard MHA is given by:

$$Memory Access = (\underbrace{bnd}_{\text{XW}} + \underbrace{bhn^2}_{\text{Softmax}} + \underbrace{d^2}_{W^O})$$
$$Operations = (bnd^2)$$

Where $b$ is the batch size, $n$ is the sequence length, and $d$ is the model dimension. As such, our arithmetic intensity (FLOPs/Memory Access) is given by:

$$O\left(\left(\frac{1}{k} + \frac{1}{bn}\right)^{-1}\right)$$

Where $k$ refers to the SRAM capacity, which is the number of tokens that can be stored in the GPU's fast on-chip memory at once. At inference time, the total arithmetic operations are kept the same, but the memory is given by $(bn^2d +nd^2)$. The arithmetic intensity is now given by:

$$O\left(\left(\frac{n}{d} + \frac{1}{b}\right)^{-1}\right)$$

Note that using the MHA architecture, we often do not want shorter sequence length or bigger model in order to increase arithmetic intensity.

<div align="center">
  <img src="../Medias/KVCache.png" width="550">
</div>

### Group Query Attention (GQA)

In multihead attentions, subweights $W_i$ are distinct. In GQA, to reduce the amount of memory KV caching takes, we choose to duplicates certain heads. For example if $h = 4$, we can choose $n_g = 2$ such that there are 2 distinct weights of $W^K$ and $W^V$. We then duplicate these weights until we reach our head counts. Formally:

$$W^K, W^V \in \mathbb{R}^{d_{model} \times (d_k n_g)}$$
$$KW^K = [K'_1,...,K'_{n_g}], VW^V = [V'_1,...,V'_{n_g}]$$

Where $K'_i, V'_i \in \mathbb{R}^{seq \times d_k}$. Using our previous example where $n_g = 2$ and $h = 4$, we then duplicates it in the following ways:

$$K' = [K'_1,K'_1, K'_2, K'_2] = [K'_1,K'_2] 
\begin{bmatrix}
I_{d_k} & I_{d_k} & 0 & 0 \\
0 & 0 & I_{d_k} & I_{d_k}
\end{bmatrix}$$

Note that this is the exact same as Broadcasting. If we choose $n_g = h$, we get our usual Multi-Head Attention, and if we choose $n_g = 1$, then we get Multi-Query Attention.

By reusing some Queries and Keys matrices, we thus cut down on the total memory access while keeping most of the expressiveness of the MHA transformer archetecture. The memory access is now given by $(bnd + bn^2 k +nd^2)$ and arithmetic intensity is given by:

$$O \left( \left(\frac{1}{d} + \frac{n}{dh} + \frac{1}{b} \right)^{-1} \right)$$

This now gives us a way to increase arithmetic intensity without the prior constraints.



### Multi-head Latent Attention (MLA)

<div align="center">
  <img src="../Medias/MLA.png" width="600">
</div>

In multi-head Latent attention, we attempt to compress our Key and Value matrices into a latent space that preserves important aspects of the 2, and then decode that latent space to get back our $W^K$ and $W^V$. Formally, we first compute:

$$L^{KV} = XW^{KV}_{\downarrow}$$

Where $W^{KV} \in \mathbb{R}^{d_{model} \times d_{latent}}$ where $d_{latent} < d_{model}$. We then "up-project" this latent space back via the following:

$$K = L^{KV}W^{K}_{\uparrow}$$
$$V = L^{KV}W^{V}_{\uparrow}$$

While this approach adds additional computes, and thus undermining the purpose of KV cache, we can actually rearrange the archetecture in the following way that minimize both:

$$Q' = XW^Q (W^{K}_{\uparrow})^T$$

We then compute:

$$K' = V' = L^{KV}$$

$$O = Attention(Q', K',  V')$$

We then multiply it back using our new matrix given by:

$$ O' = O W^{V}_{\uparrow} W^O $$

This process can be extented into our usual multi-head approach by splitting up each each $Q', K',$ and $V'$ into different heads, computing their attention, concatinating them together, and then multiply by $W^{V}_{\uparrow} W^O$. We see from this formulation that every head shares the same $L^{KV}$, which is cache and is used by all the attentions. It accomplishes the same thing as KV cache while improving both performance and cost.

### RoPE